In [ ]:
# Multi-Commodity Network Flow (4 nodes, 4 arcs, 2 commodities) — gurobipy
import gurobipy as gp
from gurobipy import GRB
nodes=[1,2,3,4]; arcs=[(1,2),(1,3),(2,4),(3,4)]; K=[1,2]
cap={(1,2):20,(1,3):15,(2,4):25,(3,4):20}
cost={(1,2,1):2,(1,2,2):3,(1,3,1):3,(1,3,2):2,(2,4,1):1,(2,4,2):2,(3,4,1):2,(3,4,2):1}
dem={(1,1):-10,(1,2):-5,(2,1):0,(2,2):0,(3,1):0,(3,2):0,(4,1):10,(4,2):5}
m=gp.Model("MultiCommodityFlow")
f=m.addVars([(i,j,k) for (i,j) in arcs for k in K], lb=0, name="f")
m.setObjective(gp.quicksum(cost[i,j,k]*f[i,j,k] for (i,j) in arcs for k in K), GRB.MINIMIZE)
for (i,j) in arcs:
    m.addConstr(gp.quicksum(f[i,j,k] for k in K) <= cap[i,j], f"Cap_{i}_{j}")
for n in nodes:
    for k in K:
        m.addConstr(gp.quicksum(f[i,j,k] for (i,j) in arcs if i==n)
                    - gp.quicksum(f[i,j,k] for (i,j) in arcs if j==n) == -dem[n,k], f"Bal_{n}_{k}")
m.optimize()
if m.status==GRB.OPTIMAL:
    for (i,j) in arcs:
        print(f"arc {i}->{j}: " + ", ".join(f"k{k}={f[i,j,k].X:g}" for k in K))
    print("Total cost =", m.ObjVal)
